In [1]:
from util import import_ragas_custom, load_env_variables_from_all_env_files

import_ragas_custom('ragas_custom_2')
load_env_variables_from_all_env_files()

Arquivos copiados com sucesso!


In [ ]:
import os
import asyncio
import nest_asyncio

from ragas.integrations.llama_index import evaluate
from ragas.run_config import RunConfig
from ragas.testset.synthesizers.testset_schema import Testset
from llama_index.llms.openai import OpenAI
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.openai import OpenAIEmbedding, OpenAIEmbeddingModelType
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.llama_api import LlamaAPI

from ragas.prompt.mixin import PromptMixin
from ragas.run_config import RunConfig

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
    StorageContext,
    load_index_from_storage,
)

from ragas.metrics import (
    context_precision,
    context_recall,
    context_entity_recall,
    NoiseSensitivity,
    ResponseRelevancy,
    answer_relevancy,
    faithfulness,
    FactualCorrectness,
    SemanticSimilarity,
    NonLLMStringSimilarity,
    RougeScore,
    StringPresence,
    ExactMatch
)

from ragas.metrics._aspect_critic import SUPPORTED_ASPECTS

In [3]:
nest_asyncio.apply()

In [4]:
DATA_PATH = 'data'
TESTSET = 'testset_openai_4omini.jsonl'
LANGUAGE = 'portuguese'
TIMEOUT = 60
RESULT_CSV = 'result_gpt4omini_llama3_70b.csv'
MODEL_OLLAMA = 'llama3.2:3b'
MODEL_GPT = 'gpt-4o-mini-2024-07-18'
MODEL_GPT_EMBEDING = OpenAIEmbeddingModelType.TEXT_EMBED_3_SMALL
CACHE_DIR = 'cache'
MODEL_LLAMA_API = 'llama3.3-70b'
PERSIST_DIR = "./cache_gpt_emb_small"

In [5]:
run_config = RunConfig(timeout=TIMEOUT)

In [ ]:
testset = Testset.from_jsonl(TESTSET).to_pandas()

print("Tamanho do dataset: ", len(testset))
print(testset.head())

Tamanho do dataset:  132
                                          user_input  \
0  Quais são as aplicações e benefícios dos No-Br...   
1  Quais são as aplicações e benefícios dos No-Br...   
2  impacto da proteção contra distúrbios de energ...   
3  De que forma a protecão contra distúrbios de e...   
4  implicações suporte técnico eficaz melhoria ex...   

                                  reference_contexts  \
0  [A EMPRESA\nOs    No-Breaks   da   CM     Coma...   
1  [A EMPRESA\nOs    No-Breaks   da   CM     Coma...   
2  [plicações de  \nmissão crítica, nas mais vari...   
3  [plicações de  \nmissão crítica, nas mais vari...   
4  [supor te técnico pode oferecer ., supor te té...   

                                           reference          synthesizer_name  
0  Os No-Breaks da CM Comandos são indicados para...  AbstractQuerySynthesizer  
1  Os No-Breaks da CM Comandos são indicados para...  AbstractQuerySynthesizer  
2  A proteção contra distúrbios de energia elétri...  Abst

In [7]:
nan_rows = testset[testset.isna().any(axis=1)]

print("Quantidade de nulos: ", len(nan_rows))
print(nan_rows)

del nan_rows

Quantidade de nulos:  0
Empty DataFrame
Columns: [user_input, reference_contexts, reference, synthesizer_name]
Index: []


In [8]:
testset = testset.dropna()

In [9]:
print("Quantidade de linhas após remoção de nulos: ", len(testset))
print(testset)

Quantidade de linhas após remoção de nulos:  132
                                            user_input  \
0    Quais são as aplicações e benefícios dos No-Br...   
1    Quais são as aplicações e benefícios dos No-Br...   
2    impacto da proteção contra distúrbios de energ...   
3    De que forma a protecão contra distúrbios de e...   
4    implicações suporte técnico eficaz melhoria ex...   
..                                                 ...   
127  Qual é a importância do termo 'BEN' no context...   
128  Qual é a função do disjuntor no processo de ma...   
129  Quais são as aplicações mais modernas do DSP n...   
130  Quais são as vantagens do modelo cliente-serve...   
131                        significado de 'Po' lti-S.O   

                                    reference_contexts  \
0    [A EMPRESA\nOs    No-Breaks   da   CM     Coma...   
1    [A EMPRESA\nOs    No-Breaks   da   CM     Coma...   
2    [plicações de  \nmissão crítica, nas mais vari...   
3    [plicações de  \n

In [10]:
testset = Testset.from_pandas(testset[:1])

In [11]:
# embeding = OpenAIEmbedding(model=MODEL_GPT_EMBEDING)
model = OpenAI(model=MODEL_GPT)

# embeding = OllamaEmbedding(model_name=MODEL)
# model = Ollama(model=MODEL)

# model = LlamaAPI(model=MODEL_LLAMA_API, api_key=os.getenv('LLAMA_API_KEY'))
embeding = OpenAIEmbedding(model=MODEL_GPT_EMBEDING)

Settings.embed_model = embeding
Settings.llm = model

In [12]:
metrics = [
    context_precision,
    context_recall,
    context_entity_recall,
    NoiseSensitivity(),
    ResponseRelevancy(),
    answer_relevancy,
    faithfulness,
    FactualCorrectness(),
    SemanticSimilarity(),
    NonLLMStringSimilarity(),
    RougeScore(),
    StringPresence(),
    ExactMatch()
]

metrics.extend(SUPPORTED_ASPECTS)

for query in metrics:
    if isinstance(query, PromptMixin):
        path = os.path.join(CACHE_DIR, query.__class__.__name__)
        if not os.path.exists(path):
            os.makedirs(path)

        try:
            prompts = query.load_prompts(path, LANGUAGE,)
            query.set_prompts(**prompts)
        except Exception:
            prompts = asyncio.run(query.adapt_prompts(LANGUAGE, None, True, True))
            query.set_prompts(**prompts)
            query.save_prompts(path)
            prompts = query.load_prompts(path, LANGUAGE)
            query.set_prompts(**prompts)


Prompt saved to cache\ContextPrecision\context_precision_prompt_portuguese.json
Prompt strings saved to cache\ContextPrecision\strings_0.2.3.json
Prompt saved to cache\ContextRecall\context_recall_prompt_portuguese.json
Prompt strings saved to cache\ContextRecall\strings_0.2.3.json
Prompt saved to cache\ContextEntityRecall\context_entity_recall_prompt_portuguese.json
Prompt strings saved to cache\ContextEntityRecall\strings_0.2.3.json
Prompt saved to cache\NoiseSensitivity\nli_statements_message_portuguese.json
Prompt strings saved to cache\NoiseSensitivity\strings_0.2.3.json
Prompt saved to cache\NoiseSensitivity\statement_prompt_portuguese.json
Prompt saved to cache\ResponseRelevancy\question_generation_portuguese.json
Prompt strings saved to cache\ResponseRelevancy\strings_0.2.3.json
Prompt saved to cache\AnswerRelevancy\question_generation_portuguese.json
Prompt strings saved to cache\AnswerRelevancy\strings_0.2.3.json
Prompt saved to cache\Faithfulness\nli_statements_message_portu

In [13]:
if not os.path.exists(PERSIST_DIR):
    documents = SimpleDirectoryReader(DATA_PATH).load_data()
    index = VectorStoreIndex.from_documents(documents)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)
    
query_engine = index.as_query_engine(request_timeout=TIMEOUT)

In [ ]:
query_engine.query("Modelos de equipamentos:")

Response(response='Os modelos de equipamentos incluem:\n\n1. Módulo No-Break 25 kW - 380/400/415V\n   - Peso: 25,0 kg\n   - Dimensões: 436 x 85 x 677 mm\n\n2. Gabinete para Módulo 25 kW - 380/400/415V\n   - Peso: 150,0 kg\n   - Dimensões: 482 x 931 x 916 mm\n\n3. Módulo No-Break 50 kW - 380/400/415V\n   - Peso: 50,0 kg\n   - Dimensões: 510 x 178 x 700 mm\n\n4. Gabinete para Módulo 50 kW - 380/400/415V\n   - Peso: 100,0 kg\n   - Dimensões: 600 x 1150 x 980 mm\n\n5. Módulo No-Break 30 kW - 200/208/220V\n   - Peso: 30,0 kg\n   - Dimensões: 510 x 178 x 700 mm\n\n6. Gabinete para Módulo 30 kW - 200/208/220V\n   - Peso: 100,0 kg\n   - Dimensões: 600 x 1150 x 980 mm\n\n7. Modelos sem baterias:\n   - 10 kVA: 680 x 415 x 835 mm, 80 kg\n   - 15 kVA: 680 x 415 x 835 mm, 80 kg\n   - 20 kVA: 830 x 415 x 940 mm, 95 kg\n   - 30 kVA: 830 x 415 x 940 mm, 95 kg\n   - 40 kVA: 830 x 415 x 940 mm, 95 kg\n\n8. Modelos com baterias internas:\n   - 10 kVA: 680 x 415 x 835 mm, 138 kg\n   - 15 kVA: 680 x 415 x 

In [ ]:
result = evaluate(
    query_engine=query_engine,
    metrics=metrics,
    dataset=testset,
    llm=model,
    embeddings=embeding,
    run_config=run_config
)

Running Query Engine:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/18 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[5]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[6]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[7]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[3]: TimeoutError()


In [16]:
result_dataframe = result.to_pandas()
result_dataframe.to_csv(RESULT_CSV)

In [17]:
print(result)

{'context_precision': 1.0000, 'context_recall': 1.0000, 'context_entity_recall': 0.5000, 'noise_sensitivity_relevant': nan, 'answer_relevancy': nan, 'faithfulness': nan, 'factual_correctness': nan, 'semantic_similarity': 0.8441, 'non_llm_string_similarity': 0.2217, 'rouge_score': 0.2703, 'string_present': 0.0000, 'exact_match': 0.0000, 'harmfulness': 1.0000, 'maliciousness': 1.0000, 'coherence': 1.0000, 'correctness': 1.0000, 'conciseness': 1.0000}
